## Introduction to the extended version of DiCE (Diverse Counterfactual Explanations)

[Mothilal et al. (2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) introduce their method of generating counterfactual explanations considering _feasibility_, and _diversity_. [Guidotti and Ruggieri (2021)](https://link.springer.com/chapter/10.1007/978-3-030-88942-5_28), claim counterfactual explanations to be robust they should be similar for similar instances when they explain. In this study, in a search to improve the quality and reliability of the counterfactual explanations _robustness_ is found to be helpful and it also introduced in the optimization function.

DiCE-Extended is built upon the [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) [(Mothilal et al. 2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) framework by introducing a robustness term in the optimization function.

## Manipulated Optimization Function

The core enhancement in DiCE-Extended is the manipulated optimization function, designed to balance proximity, diversity, and feasibility of counterfactuals. The function is formulated as:

<a id="equation-1"></a>
\begin{equation}
\tag{1}
C(x) = \underset{c_1, ..., c_k}{\text{arg min}}
\frac{1}{2} \sum_{i=1}^{k} yloss(f(c_i), y) +
\frac{\lambda_1}{k} \sum_{i=1}^{k} dist(c_i, x) -
\lambda_2 \cdot dpp\_diversity(c_1, ..., c_k) -
\frac{\lambda_3}{k} \sum_{i=1}^{k} robustness(c_i, c_i')
\end{equation}

- **Proximity Loss**: The first term that averages the distance between generated counterfactuals and the original input ensure the counterfactuals to be as close as possible to the original input.
- **Diversity Loss**: Diversity of the counterfactual explanations is aquired by determinental point process of which loss is represented by the second term and it ensures that _k_ number of counterfactual explanations are generated.
- **Robustness Loss**: [Guidotti (2024)](https://link.springer.com/article/10.1007/s10618-022-00831-6) defines robustness as necessity of similar instances being explained by similar counterfactual explanations such that if $b(x_1)=b(x_2)=y$ then an explainer $f$ should generate counterfactuals $c_1$ and $c_2$ that are similar and can explain $x_1$ and $x_2$. The robustness term that is based on [Dice-Sørensen Coefficient](https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient), is adopted from [Bonasera and Carrizosa (2024)](
https://doi.org/10.48550/arXiv.2407.00843).

\begin{equation}
\tag{2}
Robustness(c_i, c_i') = \frac{2 * \lvert c_i \cap c_i' \rvert}{\lvert c_i \rvert + \lvert c_i' \rvert}
\end{equation}


By adjusting the weights $\lambda_1$, $\lambda_2$, $\lambda_3$ counterfactual explanations can be customised by specific needs.

## Metrics and Sensitivity Analysis for Dice Extended


### 1. Robustness Metrics

#### Dice-Sørensen Coefficient

To evaluate robustness, the Dice-Sørensen coefficient measures the similarity between counterfactuals c1 and
c2 generated for similar input instances x1 and x2:

\begin{equation}
\tag{3}
Robustness(c_1, c_2) = \frac{2 * \lvert c_1 \cap c_2 \rvert}{\lvert c_1 \rvert + \lvert c_2 \rvert}
\end{equation}

where:
- $ c_1 $ and $ c_2 $ are binary vectors,
- $ \lvert c_1 \cap c_2 \rvert $: The number of shared (overlapping) features between c1 and c2,
- $ \lvert c_1 \rvert $ and $ \lvert c_2 \rvert $: The total number of features in each counterfactual.

#### Input Perturbation and Stability

Stability under input perturbation measures the solution variance when slight perturbations are introduced
to the input instance. The procedure includes the following steps:

1) **Apply Gaussian Noise:** Perturb the input $x$ by adding Gaussian noise $\delta$ to create perturbed inputs
$x'$:

\begin{equation}
\tag{4}
x' = x + \delta, \quad \delta \sim \mathcal{N}(0, \sigma^2)
\end{equation}

where $\sigma$ is the standard deviation of the noise (e.g., $\sigma = 0.01$).

2) **Generate Counterfactuals:** Generate counterfactual explanations $c_i$ for the original input $x$ and $c_i'$ for the perturbed input $x'$.

3) **Measure Stability:** Compare counterfactuals using a distance metric, such as the Euclidean distance:

\begin{equation}
\tag{5}
Stability = \frac{1}{n} \sum_{i=1}^{n} dist(c_i, c_i')
\end{equation}

where:

\begin{equation}
\tag{6}
dist(c_i, c_i') = \sqrt{\sum_{j=1}^{d} (c_{ij} - c_{ij}')^2}
\end{equation}

$n$ is the total number of input instances, $c_i$ is the counterfactual for the original input, and $c_i'$ is the counterfactual for the perturbed input.

### 2. Counterfactual Quality Measures

#### Fidelity

Fidelity measures how often generated counterfactuals successfully change the model’s prediction:

\begin{equation}
\tag{7}
Fidelity = \frac{\sum_{i=1}^{n} \mathbf{1}(f(c_i) = y_{desired})}{n}
\end{equation}

where:

- $f$: Prediction model,
- $c_i$: Counterfactual instance,
- $y_{desired}$: Target output class,
- $n$: Total number of counterfactuals.

#### Proximity

Proximity measures the average distance between counterfactuals $c_i$ and the original inputs $x_i$:

\begin{equation}
\tag{8}
Proximity = \frac{1}{n} \sum_{i=1}^{n} dist(x_i, c_i)
\end{equation}

The Manhattan distance can be used for simplicity:

\begin{equation}
\tag{9}
dist(x_i, c_i) = \sum_{j=1}^{d} \lvert x_{ij} - c_{ij} \rvert
\end{equation}

#### Diversity

Diversity measures how dissimilar the counterfactuals $c_1, c_2, c_3,\ldots,c_k$ are among themselves:

\begin{equation}
\tag{10}
Diversity = \frac{1}{k(k-1)}\sum_{i_1}^{k}\sum_{j \neq i}^{} dist(c_i, c_j)
\end{equation}

where $k$ is the number of counterfactuals.

### 3. Sensitivity Analysis

#### Objective Function with Weights

The modified loss function in DiCE-Extended is defined as in the [equation 1](#equation-1) where:

- $yloss(f(c_i), y)$: Prediction loss for counterfactual instance $c_i$ relative to the desired outcome $y$,
- $dist(c_i, x)$: Distance metric (e.g., Euclidean or Manhattan) between the counterfactual c_i and the original input $x$,
- $dpp_diversity(c_1,\ldots,c_k)$: Diversity loss term based on Determinantal Point Process (DPP),
- $Robustness(c_i,c_i')$: Robustness loss measuring similarity of counterfactuals under perturbations.

  The weights $\lambda_1, \lambda_2, \lambda_3$ control the balance between proximity, diversity, and robustness, respectively.

#### Sensitivity Analysis

To perform sensitivity analysis:

1) Vary the weights $\lambda_1, \lambda_2, \lambda_3$ systematically while ensuring:

\begin{equation}
\tag{11}
\lambda_1 + \lambda_2 + \lambda_3 = 1 (for normalization).
\end{equation}

2) Track the changes in the following metrics:

\begin{equation}
\tag{12}
P(\lambda_1, \lambda_2, \lambda_3) = Proximity,
\end{equation}

\begin{equation}
\tag{13}
D(\lambda_1, \lambda_2, \lambda_3) = Diversity,
\end{equation}

\begin{equation}
\tag{14}
R(\lambda_1, \lambda_2, \lambda_3) = Robustness,
\end{equation}

3) Measure the relationship between these metrics and the weights.



In [33]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [34]:
import numpy as np
import timeit
import random
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from torch.utils.data import DataLoader

import dice_ml_x
from dice_ml_x.utils import helpers
from dice_ml_x.utils import neuralnetworks
from dice_ml_x import DiceX

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from dice_ml_x.benchmarking import Benchmarking
datasets = ["compas-recidivism", "adult-income", "lending-club", "german-credit"]
datasets = ["lending-club"]
backends = ["TF2"]
methods = ['gaussian', 'random', 'spherical']
benchmarking = Benchmarking(datasets=datasets,
                            backends=backends,
                            perturbation_methods=methods)
benchmarking.load_and_train(batch_size=16)

Benchmarking:   0%|          | 0/1 [00:00<?, ?it/s]

continuous features are:  ['employment_years', 'num_open_credit_acc', 'annual_income', 'credit_history', 'loan_status']
target column is:  loan_status
the dataset is : lending-club, the backend is : TF2, the method is gaussian
Diverse Counterfactuals found! total time taken: 04 min 03 sec
the dataset is : lending-club, the backend is : TF2, the method is random


In [60]:
benchmarking.results

{'lending-club': {'PYT': {'accuracy': 0.17044310171198387,
   'cfs': {'gaussian':    employment_years  num_open_credit_acc  annual_income loan_grade  \
    0                10                   36       490186.7          F   
    1                10                   42       280377.1          F   
    2                 0                    3       377401.6          D   
    3                 0                   11       415134.1          D   
    
       credit_history   purpose      home addr_state  loan_status  
    0           -39.5  purchase       OWN         VT        0.397  
    1           -39.5  purchase       OWN         VT        0.386  
    2            16.9  purchase  MORTGAGE         VA        0.437  
    3            42.4  purchase  MORTGAGE         VA        0.431  ,
    'random':    employment_years  num_open_credit_acc  annual_income loan_grade  \
    0                10                   38       429221.9          F   
    1                10                   42    

In [15]:
import pickle

with open("benchmarking_results.pkl", "wb") as f:
    pickle.dump(benchmarking.results, f)